# 02 — Learned Onset Activation (EXP-016)

Trains a pure-numpy logistic-regression onset model and saves weights to
`models/onset_lr.npz`. The model outputs a per-frame onset *activation*; the
unchanged `OnsetDetector` peak picker makes the musical decision (rules-clean).

**Status: NOT the active onset path (config.onset.learned stays False).**
5-fold CV gives onset F1 0.7729 on the 277-combined set (> fusion 0.7615), but
that gain is *entirely* on the 150 extra-onset files. On the 127-corpus — which
the leaderboard shows the test set resembles (fusion lb 0.775 sits at the c127
level, not the 277 level) — the learned model trails fusion 0.778 vs 0.806.
So fusion (EXP-015) remains the submission onset detector. This notebook is kept
as validated infrastructure: feed it richer features (more ODFs, mel context)
and it may eventually beat fusion on the c127 corpus too. See EXP-016 in the log.

Workflow: **Restart & Run All** -> held-out CV F1 + subset split -> trains the
final model -> saves weights (only used if you set `config.onset.learned=True`).


In [ ]:
# Setup
import sys
from pathlib import Path

import numpy as np
import mir_eval

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.config import config
from src.data_loader import DataLoader
from src.features import FeatureExtractor
from src.detectors import OnsetDetector
from src import learned_onset as lo

config.paths.base_dir = project_root
config.onset.fusion_odfs = ("superflux", "complex")  # ODF channels = model features
fps = config.audio.sample_rate / config.audio.onset_hop_length
print("setup ok; fps =", round(fps, 2))

In [ ]:
# Data + features (ODF channels per file)
loader = DataLoader()
train = loader.load_train(project_root / "data" / "processed" / "train")
extra = loader.load_extra_onsets(project_root / "data" / "processed" / "train_extra_onsets")
all_data = {**train, **extra}
train_stems = set(train.keys())  # c127 corpus (the test set resembles this)
print(f"Onset-labelled files: {len(all_data)} (127 train + {len(extra)} extra)")

fe = FeatureExtractor()
files = []
for stem, info in all_data.items():
    y, sr = loader.load_audio(info["wav"])
    if y is None:
        continue
    files.append({
        "stem": stem,
        "chans": fe.onset_channels(y),  # (n_chan, T): superflux bands + complex
        "onsets": np.asarray(info.get("onsets") or [], dtype=float),
        "subset": "c127" if stem in train_stems else "extra",
    })
print(f"Built ODF channels for {len(files)} files")

In [ ]:
# Hyperparameters (winning CV config) + honest 5-fold held-out evaluation
CTX, LABEL_W, W_POS = 5, 1, 2.0        # context +-frames, label +-frames, positive weight
DELTA = config.onset.learned_delta      # peak-pick delta on the learned activation (0.18)
N_FOLDS, SEED = 5, 0

od = OnsetDetector()
rng = np.random.default_rng(SEED)
order = rng.permutation(len(files))
folds = np.array_split(order, N_FOLDS)

held = []
sub = {"c127": [], "extra": []}
for fi in range(N_FOLDS):
    test_idx = set(folds[fi].tolist())
    tr = [files[i] for i in range(len(files)) if i not in test_idx]
    te = [files[i] for i in range(len(files)) if i in test_idx]
    model = lo.fit([f["chans"] for f in tr], [f["onsets"] for f in tr], fps,
                   ctx=CTX, label_w=LABEL_W, w_pos=W_POS)
    for f in te:
        if f["onsets"].size == 0:
            continue
        act = model.predict_activation(f["chans"])
        peaks = od._pick(act, fps, DELTA)        # SAME hand-written picker
        est = np.array(peaks, dtype=int) / fps
        fm = mir_eval.onset.f_measure(f["onsets"], est, window=0.05)[0] if len(est) else 0.0
        held.append(fm)
        sub[f["subset"]].append(fm)
    print(f"  fold {fi+1}/{N_FOLDS} done")

print(f"\nHELD-OUT onset F1 ({N_FOLDS}-fold CV, delta={DELTA}): {np.mean(held):.4f}  "
      f"({len(held)} files)")
print("EXP-015 fusion baseline (277): 0.7615")
print(f"\nBy subset (the test set resembles c127):")
print(f"  LEARNED  c127={np.mean(sub['c127']):.4f}  extra={np.mean(sub['extra']):.4f}")
print(f"  FUSION   c127=0.8055       extra=0.7242")
print("  -> fusion wins the c127 corpus, so it stays the submission onset detector.")

In [ ]:
# Train the FINAL model on all 277 files and save weights for the submission.
# (Honest generalization estimate is the held-out CV number above, NOT a
#  re-evaluation on these training files. The test set is the true held-out.)
model = lo.fit([f["chans"] for f in files], [f["onsets"] for f in files], fps,
               ctx=CTX, label_w=LABEL_W, w_pos=W_POS, odfs=("superflux", "complex"))
out = project_root / config.onset.learned_model_path
model.save(out)
print(f"Saved model -> {out}")
print(f"  ctx={model.ctx}, odfs={model.odfs}, feat_dim={model.w.shape[0]}")
print("\nNext: in 01_pipeline.ipynb Parameters cell set  config.onset.learned = True")